In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
_base_dir = '/content/drive/My Drive/Thesis/book_recsys'
_data_dir = f'{_base_dir}/data/processed_2'
_models_dir = f'{_data_dir}/model_v1'
_eval_dir = f'{_data_dir}/eval_v1'

user_index = f'{_models_dir}/user_index.csv'
imtem_index = f'{_models_dir}/cb_sbert_book_index.csv'
history_db = f'{_models_dir}/user_history.db'

sbert_item_matrix = f'{_models_dir}/cb_sbert_matrix.npy'
sbert_user_matrix = f'{_models_dir}/cb_sbert_user_matrix.npy'
sbert_item_faiss_index = f'{_models_dir}/cb_sbert_matrix.index'

ials_user_matrix = f'{_models_dir}/cf_als_user_profiles.npy'
ials_item_matrix = f'{_models_dir}/cf_als_item_matrix.npy'
ials_item_faiss_index = f'{_models_dir}/cf_als_item_matrix.index'

test_file = f'{_data_dir}/test_main.csv'
ground_truth_file = f'{_eval_dir}/test_ground_truth.json'

cbf_predictions = f'{_eval_dir}/cbf_prediction.json'
cf_predictions = f'{_eval_dir}/cf_test_predictions.json'

cbf_predictions_v2 = f'{_eval_dir}/cbf_test_predictions_v2.json'

raw_books = f'{_data_dir}/books_filtered.parquet'

In [3]:
import pandas as pd
import gc
gc.collect()

# ---------------------------------------------------------------------------
# CONFIG - update the path if the file lives somewhere else
# ---------------------------------------------------------------------------
# raw_books = "books_filtered.parquet"

# ---------------------------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------------------------
df = pd.read_parquet(f'{_data_dir}/books_cleaned.parquet')
total_books = len(df)

print("=" * 70)
print(f"TOTAL BOOKS IN DATASET: {total_books:,}")
print("=" * 70)

# ---------------------------------------------------------------------------
# 2. List all columns (fields) in the file
# ---------------------------------------------------------------------------
print("\n--- ALL FIELDS (COLUMNS) IN THE FILE ---\n")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:>3}. {col}")

# ---------------------------------------------------------------------------
# 3. Missing metadata analysis
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("MISSING METADATA ANALYSIS")
print("=" * 70)

missing_stats = (
    df.isnull()
    .sum()
    .reset_index()
    .rename(columns={"index": "field", 0: "missing_count"})
)
missing_stats["missing_pct"] = (
    missing_stats["missing_count"] / total_books * 100
).round(2)
missing_stats = missing_stats.sort_values("missing_count", ascending=False)

# Also count empty strings as "missing" for object columns
for col in df.select_dtypes(include="object").columns:
    empty_str_count = (df[col].astype(str).str.strip() == "").sum()
    if empty_str_count > 0:
        idx = missing_stats.loc[missing_stats["field"] == col].index
        if len(idx):
            orig = missing_stats.loc[idx[0], "missing_count"]
            combined = orig + empty_str_count
            missing_stats.loc[idx[0], "missing_count"] = combined
            missing_stats.loc[idx[0], "missing_pct"] = round(
                combined / total_books * 100, 2
            )

missing_stats = missing_stats.sort_values("missing_count", ascending=False)

print(f"\n{'Field':<30} {'Missing':>10} {'Pct (%)':>10}")
print("-" * 52)
for _, row in missing_stats.iterrows():
    print(f"{row['field']:<30} {int(row['missing_count']):>10,} {row['missing_pct']:>9.2f}%")

# Summary: books with at least one missing field
books_with_any_missing = df.isnull().any(axis=1).sum()
print(f"\nBooks with at least 1 missing field: {books_with_any_missing:,} "
      f"({books_with_any_missing / total_books * 100:.2f}%)")

# ---------------------------------------------------------------------------
# 4. Genre diversity analysis
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("GENRE DIVERSITY ANALYSIS")
print("=" * 70)

# Try common column names for genre
genre_col = None
for candidate in ["genre", "genres", "Genre", "Genres", "category", "categories"]:
    if candidate in df.columns:
        genre_col = candidate
        break

if genre_col is None:
    print("\n[WARNING] No genre/category column found. Available columns:")
    print(f"  {list(df.columns)}")
else:
    # Genres may be stored as lists, comma-separated strings, or single values
    genre_series = df[genre_col].dropna()

    all_genres = []
    for val in genre_series:
        if isinstance(val, list):
            all_genres.extend([str(g).strip() for g in val if str(g).strip()])
        else:
            # try splitting by comma or pipe
            parts = str(val).replace("|", ",").split(",")
            all_genres.extend([g.strip() for g in parts if g.strip()])

    genre_counts = pd.Series(all_genres).value_counts()
    unique_genres = len(genre_counts)

    print(f"\nUnique genres found: {unique_genres}")
    print(f"Total genre tags across all books: {len(all_genres):,}")
    print(f"\nTop 20 genres:")
    print(f"  {'Genre':<40} {'Count':>8}")
    print("  " + "-" * 50)
    for genre, count in genre_counts.head(20).items():
        print(f"  {genre:<40} {count:>8,}")

    if unique_genres > 20:
        print(f"\n  ... and {unique_genres - 20} more genres")

# ---------------------------------------------------------------------------
# 5. Sample rows (5-10 rows)
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("SAMPLE ROWS FROM THE FILE")
print("=" * 70)

sample_size = min(10, total_books)
sample = df.sample(n=sample_size, random_state=42)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.width", 200)

print(f"\nShowing {sample_size} random rows:\n")
print(sample.to_string(index=False))

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

TOTAL BOOKS IN DATASET: 1,173,713

--- ALL FIELDS (COLUMNS) IN THE FILE ---

    1. book_id
    2. title
    3. title_without_series
    4. author_name
    5. genres
    6. description
    7. publisher
    8. publication_year
    9. publication_month
   10. publication_day
   11. num_pages
   12. is_ebook
   13. language_code
   14. average_rating
   15. ratings_count
   16. image_url

MISSING METADATA ANALYSIS

Field                             Missing    Pct (%)
----------------------------------------------------
language_code                     447,334     38.11%
publication_day                   420,047     35.79%
publication_month                 357,017     30.42%
num_pages                         301,782     25.71%
publisher                         291,513     24.84%
publication_year                  255,933     21.81%
description                       134,764     11.48%
genres                             52,473      4.47%
author_name                           245      0.02%
r

In [4]:
import pandas as pd
import ast
import gc
gc.collect()

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
df = pd.read_parquet(raw_books)

print("=" * 70)
print("BEFORE CLEANING")
print("=" * 70)
print(f"Total books: {len(df):,}")
print(f"\nSample genres (raw):")
print(df["genres"].head(5).to_string())


# ---------------------------------------------------------------------------
# CLEANING FUNCTION
# ---------------------------------------------------------------------------
def clean_genres(raw_value):
    """
    Input format (string representation of a dict):
      "{'history, historical fiction, biography': 259.0,
        'fiction': 5.0,
        'fantasy, paranormal': None, ...}"

    Rules:
      - Remove entries where the value is None
      - For non-None entries, keep only the genre name (key), drop the number
    Returns:
      A comma-separated string of genre names, or None if nothing remains.
    """
    if pd.isna(raw_value):
        return None

    # Parse the string into a real dict
    try:
        genre_dict = ast.literal_eval(str(raw_value))
    except (ValueError, SyntaxError):
        return None

    if not isinstance(genre_dict, dict):
        return None

    # Keep only keys whose value is not None
    valid_genres = [genre for genre, value in genre_dict.items() if value is not None]

    if not valid_genres:
        return None

    return ", ".join(valid_genres)


# ---------------------------------------------------------------------------
# APPLY
# ---------------------------------------------------------------------------
df["genres"] = df["genres"].apply(clean_genres)

print("\n" + "=" * 70)
print("AFTER CLEANING")
print("=" * 70)
print(f"\nSample genres (cleaned):")
print(df["genres"].head(10).to_string())

# Quick stats
non_null = df["genres"].notna().sum()
null_count = df["genres"].isna().sum()
print(f"\nBooks with genres: {non_null:,}")
print(f"Books without any genre (all were None): {null_count:,}")

# Save
output_path = f"{_data_dir}books_cleaned.parquet"
df.to_parquet(output_path, index=False)
print(f"\nSaved to: {output_path}")
print("=" * 70)

BEFORE CLEANING
Total books: 1,173,713

Sample genres (raw):
0    {'history, historical fiction, biography': Non...
1    {'history, historical fiction, biography': Non...
2    {'history, historical fiction, biography': Non...
3    {'history, historical fiction, biography': 1.0...
4    {'history, historical fiction, biography': 38....

AFTER CLEANING

Sample genres (cleaned):
0    fiction, fantasy, paranormal, mystery, thrille...
1           fiction, mystery, thriller, crime, romance
2    fiction, fantasy, paranormal, children, young-...
3    history, historical fiction, biography, non-fi...
4    history, historical fiction, biography, fictio...
5    history, historical fiction, biography, fictio...
6    history, historical fiction, biography, fictio...
7    history, historical fiction, biography, fictio...
8                                          non-fiction
9    history, historical fiction, biography, fictio...

Books with genres: 1,121,240
Books without any genre (all were None): 5